In [1]:
# ====================== 第一段：安装依赖 + 导入库 ======================
# 运行前打开您的终端（Terminal）或命令提示符（CMD），直接复制并运行以下命令下载模型库：
# pip install numpy pandas matplotlib seaborn shap xgboost networkx scikit-learn statsmodels openpyxl
# 如果您使用的是国内网络环境，建议加上清华镜像源以加快下载速度：
# pip install numpy pandas matplotlib seaborn shap xgboost networkx scikit-learn statsmodels openpyxl -i https://pypi.tuna.tsinghua.edu.cn/simple

import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.colors as mcolors
import matplotlib.font_manager as fm
import seaborn as sns
import shap
import xgboost as xgb
import networkx as nx
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import VarianceThreshold
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, train_test_split
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import statsmodels.api as sm
from mpl_toolkits.axes_grid1 import make_axes_locatable
import matplotlib.lines as mlines

print("✅ 第一段：库导入完成")

✅ 第一段：库导入完成


In [2]:
# ====================== 第二段：全局配置 + 字体设置 ======================
CONFIG = {
    "data_file": "20260312_Matched_cleaned.xlsx",
    "output_dir": "PM25_纯街景_运行结果",
    "dpi": 200,
    "formats": ["png", "pdf"],
    
    "font_family_en": "Times New Roman",
    "font_family_zh": "SimSun",
    "label_fontsize": 14,
    "title_fontsize": 16,
    "tick_fontsize": 14,
    
    "fig1_kde_linewidth": 1.5,
    "fig1_hist_bins": 50,
    "fig1_train_color": "#A9A9A9",
    "fig1_test_color": "#9F3E3F",
    "fig1_scatter_s": 30,
    "fig1_scatter_edgecolor": "white",
    "fig1_scatter_linewidth": 0.5,
    
    "fig2_scatter_s": 12,
    "fig2_bar_color": "#C3D3F2",
    
    "fig3_scatter_s": 20,
    "fig3_curve_color": "#FF451B",
    "fig3_curve_linewidth": 1.5,
    "fig3_cbar_fontsize": 10,
    "fig3_positive_color": "#DEF4F1",
    "fig3_negative_color": "#E6DADA",
    
    "fig4_main_color": "#71BCB1",
    "fig4_inter_color": "#9F6566",
    
    "fig5_max_features": 18,
    "fig7_max_features": 18,
    "fig8_max_display": 18,

    "fig5_label_fontsize": 16,
    "fig5_tick_fontsize": 11,
    "fig5_cbar_fontsize": 16,
    
    "fig7_node_color": "#48A597",
    "fig7_edge_color": "#9C1A1C",

    "fig8_figsize": (10, 6),
    
    "fig9_max_instances": 50,
    "fig9_figsize": (16, 4),
    "fig9_color_pos": "#9C1A1C",
    "fig9_color_neg": "#48A597",

    "fig10_line_color_q1": "#2A6F97",
    "fig10_line_color_med": "#48A597",
    "fig10_line_color_q3": "#9C1A1C",
    
    "shap_cmap": mcolors.LinearSegmentedColormap.from_list("custom_cmap", [ "#48A597", "#FFFFFF", "#9C1A1C"]),
    "random_state": 42
}

# 字体配置
available_fonts = [f.name for f in fm.fontManager.ttflist]
en_font = CONFIG["font_family_en"]
zh_font = CONFIG["font_family_zh"]

if en_font not in available_fonts:
    print(f"警告：系统缺失英文字体 '{en_font}'，将使用备用字体。")
if zh_font not in available_fonts:
    print(f"警告：系统缺失中文字体 '{zh_font}'，中文可能无法正常显示。")

font_fallback_list = [en_font, zh_font, 'sans-serif']
plt.rcParams['font.sans-serif'] = font_fallback_list
plt.rcParams['font.serif'] = font_fallback_list
plt.rcParams['axes.unicode_minus'] = False

sns.set_theme(style="ticks", rc={
    "font.sans-serif": font_fallback_list,
    "font.serif": font_fallback_list,
    "font.family": font_fallback_list,
    "axes.unicode_minus": False
})

print("✅ 第二段：配置与字体设置完成")

✅ 第二段：配置与字体设置完成


In [3]:
# ====================== 第三段：辅助函数 ======================
def save_figure(fig, filename_base, subfolder=None):
    save_dir = CONFIG["output_dir"]
    if subfolder:
        save_dir = os.path.join(save_dir, subfolder)
    os.makedirs(save_dir, exist_ok=True)
    
    safe_filename_base = str(filename_base).replace('/', '_').replace('\\', '_')
    
    for fmt in CONFIG["formats"]:
        filepath = os.path.join(save_dir, f"{safe_filename_base}.{fmt}")
        fig.savefig(filepath, dpi=CONFIG["dpi"], format=fmt, bbox_inches='tight')
    plt.close(fig)

print("✅ 第三段：辅助函数定义完成")

✅ 第三段：辅助函数定义完成


In [4]:
# ====================== 第四段：核心 XGBoost + SHAP 分析类 ======================
class XGBoostXAIAnalyzer:
    def __init__(self):
        self.random_state = CONFIG["random_state"]
        self.cv = 5
        self.model = None
        self.pipeline = None
        self.explainer = None
        self.shap_values_obj = None
        self.shap_interaction_values = None
        self.X_train_processed = None
        
    def _preprocess_steps(self, scale: bool):
        steps = [
            ('imputer', SimpleImputer(strategy="median")),
            ('var', VarianceThreshold(threshold=0.0)),
        ]
        if scale:
            steps.append(('scaler', StandardScaler()))
        return steps

    def _fit_random(self, name, reg, param_dist, X_train, y_train, scale: bool, n_iter=20):
        pipe = Pipeline(self._preprocess_steps(scale=scale) + [('reg', reg)])
        rscv = RandomizedSearchCV(
            pipe, param_distributions=param_dist, n_iter=n_iter,
            scoring='r2', cv=self.cv, n_jobs=-1, random_state=self.random_state, verbose=0
        )
        rscv.fit(X_train, y_train)
        print(f"{name} 最优参数: {rscv.best_params_}")
        return name, rscv.best_estimator_

    def train_xgboost(self, X_train, y_train):
        xgb_reg = xgb.XGBRegressor(objective='reg:squarederror', random_state=self.random_state, tree_method='hist', n_jobs=-1)
        
        param_dist = {
            'reg__n_estimators': [300, 500, 800, 1000],
            'reg__max_depth': [3, 4, 6, 8],
            'reg__learning_rate': [0.01, 0.05, 0.1],
            'reg__subsample': [0.7, 0.85, 1.0],
            'reg__colsample_bytree': [0.7, 0.85, 1.0],
            'reg__reg_alpha': [0, 0.1, 0.5],
            'reg__reg_lambda': [1.0, 1.5, 2.0]
        }
        
        _, best_pipeline = self._fit_random("XGBoost", xgb_reg, param_dist, X_train, y_train, scale=False, n_iter=28)
        self.pipeline = best_pipeline
        self.model = best_pipeline.named_steps['reg']
        return self.pipeline

    def prepare_data(self):
        print("正在读取数据表...")
        file_path = CONFIG["data_file"]
    
        if file_path.endswith(".csv"):
            df = pd.read_csv(file_path)
        elif file_path.endswith((".xlsx", ".xls")):
            df = pd.read_excel(file_path)
        else:
            raise ValueError("不支持的文件格式！仅支持 .csv / .xlsx / .xls")
    
        self.y = df["PM25"]
        self.X = df.iloc[:, -26:]
    
        self.feature_names = self.X.columns.tolist()
    
        self.X_train, self.X_test, self.y_train, self.y_test = train_test_split(
            self.X, self.y, test_size=0.2, random_state=self.random_state
        )
        print(f"数据读取完毕。特征数量: {self.X.shape[1]}")
        
    def calculate_shap(self):
        print("正在计算 SHAP 值 (交互作用计算较耗时)...")
        preprocessor = Pipeline(self.pipeline.steps[:-1])
        X_train_transformed = preprocessor.transform(self.X_train)
        self.X_train_processed = pd.DataFrame(X_train_transformed, columns=self.feature_names)
        
        self.explainer = shap.TreeExplainer(self.model)
        self.shap_values_obj = self.explainer(self.X_train_processed, check_additivity=False)
        self.shap_interaction_values = self.explainer.shap_interaction_values(self.X_train_processed)
        print("SHAP 计算完成。")

    # 下面是所有绘图函数，我全部保留，不拆分
    def plot_figure_1(self):
        print("正在绘制图1：模型评估组合图...")
        y_train_pred = self.pipeline.predict(self.X_train)
        y_test_pred = self.pipeline.predict(self.X_test)
        
        r2_train = r2_score(self.y_train, y_train_pred)
        rmse_train = np.sqrt(mean_squared_error(self.y_train, y_train_pred))
        r2_test = r2_score(self.y_test, y_test_pred)
        rmse_test = np.sqrt(mean_squared_error(self.y_test, y_test_pred))
        
        mae_train = mean_absolute_error(self.y_train, y_train_pred)
        mae_test = mean_absolute_error(self.y_test, y_test_pred)
        res_train = y_train_pred - self.y_train
        res_test = y_test_pred - self.y_test

        fig = plt.figure(figsize=(8, 10))
        gs = gridspec.GridSpec(3, 2, width_ratios=[4, 1], height_ratios=[1, 4, 1.5], wspace=0.05, hspace=0.05)
        
        ax_main = fig.add_subplot(gs[1, 0])
        ax_main.scatter(self.y_train, y_train_pred, color=CONFIG["fig1_train_color"],
                        s=CONFIG["fig1_scatter_s"], edgecolor=CONFIG["fig1_scatter_edgecolor"],
                        linewidth=CONFIG["fig1_scatter_linewidth"], label="Train data", alpha=0.8, zorder=2)
        ax_main.scatter(self.y_test, y_test_pred, color=CONFIG["fig1_test_color"],
                        s=CONFIG["fig1_scatter_s"], edgecolor=CONFIG["fig1_scatter_edgecolor"],
                        linewidth=CONFIG["fig1_scatter_linewidth"], label="Test data", alpha=0.8, zorder=2)
        
        xlims = ax_main.get_xlim()
        ylims = ax_main.get_ylim()
        
        ax_main.plot(xlims, ylims, 'k--', zorder=5)
        m, b = np.polyfit(self.y_test, y_test_pred, 1)
        y_fit_start, y_fit_end = m * xlims[0] + b, m * xlims[1] + b
        ax_main.plot(xlims, [y_fit_start, y_fit_end], color='black', linewidth=2, label="Fitted line", zorder=5)
        
        ax_main.set_xlim(xlims)
        ax_main.set_ylim(ylims)
        
        ax_main.text(0.5, 0.96, "XGBoost", transform=ax_main.transAxes, ha='center', va='top',
                     fontsize=CONFIG["title_fontsize"]+4, fontweight='bold', zorder=10)
        
        metrics_text = f"Train $R^2$ = {r2_train:.2f}, $RMSE$ = {rmse_train:.2f}\nTest $R^2$ = {r2_test:.2f}, $RMSE$ = {rmse_test:.2f}"
        ax_main.text(0.05, 0.85, metrics_text, transform=ax_main.transAxes, va='top', ha='left',
                     fontsize=CONFIG["title_fontsize"]-2, color="darkred",
                     bbox=dict(facecolor='white', alpha=0.8, edgecolor='none', pad=3), zorder=10)
                     
        ax_main.legend(loc="lower right", frameon=False, fontsize=CONFIG["label_fontsize"])
        ax_main.grid(True, linestyle='--', alpha=0.5, color='lightgray')

        ax_top = fig.add_subplot(gs[0, 0], sharex=ax_main)
        sns.histplot(self.y_train, color=CONFIG["fig1_train_color"], bins=CONFIG["fig1_hist_bins"],
                     ax=ax_top, alpha=0.5, stat="density", element="bars", edgecolor="white", linewidth=0.3)
        sns.histplot(self.y_test, color=CONFIG["fig1_test_color"], bins=CONFIG["fig1_hist_bins"],
                     ax=ax_top, alpha=0.5, stat="density", element="bars", edgecolor="white", linewidth=0.3)
        sns.kdeplot(self.y_train, color=CONFIG["fig1_train_color"], ax=ax_top,
                    linewidth=CONFIG["fig1_kde_linewidth"], zorder=10, cut=0)
        sns.kdeplot(self.y_test, color=CONFIG["fig1_test_color"], ax=ax_top,
                    linewidth=CONFIG["fig1_kde_linewidth"], zorder=10, cut=0)
        ax_top.axis('off')

        ax_right = fig.add_subplot(gs[1, 1], sharey=ax_main)
        sns.histplot(y=y_train_pred, color=CONFIG["fig1_train_color"], bins=CONFIG["fig1_hist_bins"],
                     ax=ax_right, alpha=0.5, stat="density", element="bars", edgecolor="white", linewidth=0.3)
        sns.histplot(y=y_test_pred, color=CONFIG["fig1_test_color"], bins=CONFIG["fig1_hist_bins"],
                     ax=ax_right, alpha=0.5, stat="density", element="bars", edgecolor="white", linewidth=0.3)
        sns.kdeplot(y=y_train_pred, color=CONFIG["fig1_train_color"], ax=ax_right,
                    linewidth=CONFIG["fig1_kde_linewidth"], zorder=10, cut=0)
        sns.kdeplot(y=y_test_pred, color=CONFIG["fig1_test_color"], ax=ax_right,
                    linewidth=CONFIG["fig1_kde_linewidth"], zorder=10, cut=0)
        ax_right.axis('off')

        ax_res = fig.add_subplot(gs[2, 0], sharex=ax_main)
        ax_res.scatter(self.y_train, res_train, color=CONFIG["fig1_train_color"],
                       s=CONFIG["fig1_scatter_s"], edgecolor=CONFIG["fig1_scatter_edgecolor"], linewidth=CONFIG["fig1_scatter_linewidth"], alpha=0.8)
        ax_res.scatter(self.y_test, res_test, color=CONFIG["fig1_test_color"],
                       s=CONFIG["fig1_scatter_s"], edgecolor=CONFIG["fig1_scatter_edgecolor"], linewidth=CONFIG["fig1_scatter_linewidth"], alpha=0.8)
        ax_res.plot(xlims, [0, 0], color='black', linewidth=1.5, zorder=5)
        ax_res.grid(True, linestyle='--', alpha=0.5, color='lightgray')
        
        res_text = f"MAE (Train) = {mae_train:.3f}\nMAE (Test) = {mae_test:.3f}"
        ax_res.text(0.95, 0.95, res_text, transform=ax_res.transAxes, fontsize=CONFIG["label_fontsize"],
                    verticalalignment='top', horizontalalignment='right',
                    bbox=dict(facecolor='white', alpha=0.8, edgecolor='none', pad=3), zorder=6)
        
        save_figure(fig, "Fig1_Prediction_Residuals")

    def plot_figure_2(self):
        print("正在绘制图2：全局特征贡献度分析图...")
        fig, ax1 = plt.subplots(figsize=(10, 8))
        
        mean_abs_shap = np.abs(self.shap_values_obj.values).mean(axis=0)
        sort_inds = np.argsort(mean_abs_shap)
        sorted_features = [self.feature_names[i] for i in sort_inds]
        sorted_mean_shap = mean_abs_shap[sort_inds]
        total_shap_sum = np.sum(mean_abs_shap)
        
        shap_vals = self.shap_values_obj.values[:, sort_inds]
        feat_vals = self.X_train_processed.values[:, sort_inds]
        y_pos = np.arange(len(sorted_features))
        
        ax2 = ax1.twiny()
        ax2.barh(y_pos, sorted_mean_shap, color=CONFIG["fig2_bar_color"], align='center', alpha=0.8, height=0.6, zorder=2)
        
        cmap = CONFIG["shap_cmap"]
        for i in range(len(sorted_features)):
            row_shap = shap_vals[:, i]
            row_feat = feat_vals[:, i]
            feat_min, feat_max = np.min(row_feat), np.max(row_feat)
            row_feat_norm = (row_feat - feat_min) / (feat_max - feat_min) if feat_max > feat_min else np.zeros_like(row_feat)
                
            jitter = np.random.normal(0, 0.1, size=len(row_shap))
            scatter = ax1.scatter(row_shap, np.repeat(i, len(row_shap)) + jitter,
                                  c=row_feat_norm, cmap=cmap, s=CONFIG["fig2_scatter_s"], alpha=0.8, edgecolors='none', zorder=4)

        max_mean_val = np.max(sorted_mean_shap)
        ax2.set_xlim(0, max_mean_val * 1.2)
        
        for i, v in enumerate(sorted_mean_shap):
            pct = (v / total_shap_sum) * 100
            offset = max_mean_val * 0.01
            ax1.text(v + offset, i, f"{pct:.1f}%", va='center', ha='left', fontsize=CONFIG["label_fontsize"],
                     transform=ax2.transData, zorder=10, bbox=dict(facecolor='white', alpha=0.8, edgecolor='none', pad=1))

        ax1.set_zorder(ax2.get_zorder() + 1)
        ax1.patch.set_visible(False)
        
        ax1.set_yticks(y_pos)
        ax1.set_yticklabels(sorted_features, fontsize=CONFIG["tick_fontsize"])
        ax1.set_xlabel("SHAP value (impact on model output)", fontsize=CONFIG["label_fontsize"])
        ax2.set_xlabel("Mean Absolute SHAP Value", fontsize=CONFIG["label_fontsize"])
        ax1.grid(True, axis='x', linestyle='--', alpha=0.4)
        
        divider = make_axes_locatable(ax1)
        cax = divider.append_axes("right", size="3%", pad=0.1)
        
        sm_map = plt.cm.ScalarMappable(cmap=cmap, norm=plt.Normalize(vmin=0, vmax=1))
        sm_map.set_array([])
        cbar = fig.colorbar(sm_map, cax=cax)
        cbar.set_ticks([0, 1])
        cbar.set_ticklabels(['Low', 'High'])
        cbar.set_label('Feature value', rotation=270, labelpad=15, fontsize=CONFIG["label_fontsize"])
        
        save_figure(fig, "Fig2_Global_Contribution")

    def plot_figure_3(self):
        print("正在绘制图3：单特征偏依赖图...")
        mean_abs_shap = np.abs(self.shap_values_obj.values).mean(axis=0)
        sort_inds_desc = np.argsort(mean_abs_shap)[::-1]
        folder_name = "Fig3_Dependence_Plots"
        
        from tqdm import tqdm
        for rank, feat_idx in tqdm(enumerate(sort_inds_desc), total=len(sort_inds_desc), desc="图3 进度"):
            feature_name = self.feature_names[feat_idx]
            fig, ax = plt.subplots(figsize=(6, 5))
            
            x_vals = self.X_train_processed[feature_name].values
            y_vals = self.shap_values_obj.values[:, feat_idx]
            c_vals = y_vals
    
            scatter = ax.scatter(x_vals, y_vals, c=c_vals, cmap=CONFIG["shap_cmap"],
                                 s=CONFIG["fig3_scatter_s"], alpha=0.9, edgecolors='none', zorder=4)
            
            sorted_indices = np.argsort(x_vals)
            x_sorted = x_vals[sorted_indices]
            y_sorted = y_vals[sorted_indices]
            
            lowess = sm.nonparametric.lowess(y_sorted, x_sorted, frac=0.3)
            ax.plot(lowess[:, 0], lowess[:, 1], color=CONFIG["fig3_curve_color"],
                    linewidth=CONFIG["fig3_curve_linewidth"], alpha=0.9, label="Lowess curve", zorder=5)
            
            window_size = max(5, int(len(x_sorted) * 0.1))
            rolling_std = pd.Series(y_sorted - lowess[:, 1]).rolling(window=window_size, min_periods=1, center=True).std().fillna(0).values
            
            upper_raw = lowess[:, 1] + rolling_std
            lower_raw = lowess[:, 1] - rolling_std
            
            upper_smoothed = sm.nonparametric.lowess(upper_raw, x_sorted, frac=0.3)[:, 1]
            lower_smoothed = sm.nonparametric.lowess(lower_raw, x_sorted, frac=0.3)[:, 1]
            
            ax.fill_between(lowess[:, 0], lower_smoothed, upper_smoothed,
                            color=CONFIG["fig3_curve_color"], alpha=0.15, edgecolor='none', linewidth=0, zorder=2)
            
            x_min, x_max = ax.get_xlim()
            y_min, y_max = ax.get_ylim()
    
            sign_changes = np.diff(np.sign(lowess[:, 1]))
            zero_crossings_idx = np.where(sign_changes != 0)[0]
            
            cross_points = []
            for idx in zero_crossings_idx:
                x1, y1 = lowess[idx, 0], lowess[idx, 1]
                x2, y2 = lowess[idx+1, 0], lowess[idx+1, 1]
                
                if y1 == 0:
                    x_c = x1
                elif y2 == 0:
                    x_c = x2
                else:
                    x_c = x1 - y1 * (x2 - x1) / (y2 - y1)
                
                if not cross_points or abs(cross_points[-1] - x_c) > 1e-5:
                    cross_points.append(x_c)
    
            boundaries = [x_min] + cross_points + [x_max]
            added_pos_label = False
            added_neg_label = False
    
            for i in range(len(boundaries) - 1):
                x_start = boundaries[i]
                x_end = boundaries[i+1]
                mid_x = (x_start + x_end) / 2
                
                nearest_idx = np.abs(lowess[:, 0] - mid_x).argmin()
                mid_y = lowess[nearest_idx, 1]
    
                if mid_y > 0:
                    label = "Positive" if not added_pos_label else None
                    ax.fill_between([x_start, x_end], 0, y_max, color=CONFIG["fig3_positive_color"], alpha=0.5, zorder=1, label=label)
                    added_pos_label = True
                else:
                    label = "Negative" if not added_neg_label else None
                    ax.fill_between([x_start, x_end], y_min, 0, color=CONFIG["fig3_negative_color"], alpha=0.5, zorder=1, label=label)
                    added_neg_label = True
    
            y_range = y_max - y_min
            for zc_idx, x_cross in enumerate(cross_points):
                ax.axvline(x=x_cross, color='#D32F2F', linestyle='--', linewidth=1.2, alpha=0.8, zorder=3)
                ax.scatter(x_cross, 0, color='#D32F2F', s=25, zorder=6)
                
                y_offset = 0.08 * y_range if zc_idx % 2 == 0 else -0.08 * y_range
                va_align = 'bottom' if zc_idx % 2 == 0 else 'top'
                
                ax.text(x_cross, y_offset, f"{x_cross:.2f}", va=va_align, ha='center',
                        fontsize=CONFIG["title_fontsize"], color='#D32F2F',
                        bbox=dict(facecolor=(1, 1, 1, 0.7), edgecolor='none', pad=2), zorder=6)
    
            ax.set_xlim(x_min, x_max)
            ax.set_ylim(y_min, y_max)
    
            ax.set_xlabel("Feature value", fontsize=CONFIG["label_fontsize"])
            ax.set_ylabel("SHAP", fontsize=CONFIG["label_fontsize"])
            ax.set_title(feature_name, fontsize=CONFIG["title_fontsize"])
            ax.grid(True, linestyle='--', alpha=0.3)
            ax.axhline(0, color='gray', linestyle='--', alpha=0.5, zorder=2)
            
            ax.legend(loc='best', fontsize=CONFIG["label_fontsize"] - 2, framealpha=0, edgecolor='none')
            
            divider = make_axes_locatable(ax)
            cax = divider.append_axes("right", size="5%", pad=0.0)
            cbar = fig.colorbar(scatter, cax=cax)
            
            cbar.set_ticks([c_vals.min(), c_vals.max()])
            cbar.set_ticklabels(['Low', 'High'])
            cbar.set_label('SHAP value', rotation=270, labelpad=15, fontsize=CONFIG["fig3_cbar_fontsize"] + 2)
            cbar.ax.tick_params(labelsize=CONFIG["fig3_cbar_fontsize"])
            
            clean_feature_name = re.sub(r'[^\w]', '', feature_name)
            filename = f"{rank+1:02d}_{clean_feature_name}_Dependence"
            save_figure(fig, filename, subfolder=folder_name)
            plt.close(fig)

    def plot_figure_4(self):
        print("正在绘制图4：主效应与交互效应对比图...")
        num_features = len(self.feature_names)
        main_effects = np.zeros(num_features)
        inter_effects = np.zeros(num_features)
        
        for i in range(num_features):
            main_effects[i] = np.abs(self.shap_interaction_values[:, i, i]).mean()
            mask = np.ones(num_features, dtype=bool)
            mask[i] = False
            inter_effects[i] = np.abs(self.shap_interaction_values[:, i, mask]).sum(axis=1).mean()
            
        total_effects = main_effects + inter_effects
        sort_inds = np.argsort(total_effects)[::-1]
        
        top_names = [self.feature_names[i] for i in sort_inds]
        top_main = main_effects[sort_inds]
        top_inter = inter_effects[sort_inds]
        
        fig_width = max(10, num_features * 0.8)
        fig, ax = plt.subplots(figsize=(fig_width, 6))
        x = np.arange(len(top_names))
        width = 0.4
        
        rects1 = ax.bar(x - width/2, top_main, width, label='Main effect (Mean |SHAP|)', color=CONFIG["fig4_main_color"])
        rects2 = ax.bar(x + width/2, top_inter, width, label='Interaction (sum over others)', color=CONFIG["fig4_inter_color"])
        
        max_height = max(np.max(top_main), np.max(top_inter))
        ax.set_ylim(0, max_height * 1.15)
        
        overlap_threshold = max_height * 0.05

        for i in range(len(top_names)):
            h1 = top_main[i]
            h2 = top_inter[i]
            
            text1 = f'{h1:.3f}'
            text2 = f'{h2:.3f}'
            
            offset_1, offset_2 = 2, 2
            va_1, va_2 = 'bottom', 'bottom'
            
            if abs(h1 - h2) < overlap_threshold:
                if h1 >= h2:
                    offset_1 = 12
                    offset_2 = 1
                else:
                    offset_1 = 1
                    offset_2 = 12

            ax.annotate(text1, xy=(x[i] - width/2, h1), xytext=(0, offset_1), textcoords="offset points", ha='center', va=va_1, fontsize=9)
            ax.annotate(text2, xy=(x[i] + width/2, h2), xytext=(0, offset_2), textcoords="offset points", ha='center', va=va_2, fontsize=9)
        
        ax.set_ylabel('Magnitude (Mean |SHAP|)', fontsize=CONFIG["label_fontsize"])
        ax.set_title('All Features: Main vs Interaction', fontsize=CONFIG["title_fontsize"])
        ax.set_xticks(x)
        ax.set_xticklabels(top_names, fontsize=CONFIG["tick_fontsize"], rotation=90)
        
        ax.legend(fontsize=CONFIG["label_fontsize"], frameon=False)
        ax.grid(True, axis='y', linestyle=':', alpha=0.6)
        
        save_figure(fig, "Fig4_Main_vs_Interaction_All")

    def plot_figure_5(self):
        print("正在绘制图5：特征交互复合矩阵图...")
        mean_abs_shap = np.abs(self.shap_values_obj.values).mean(axis=0)
        
        num_top = min(CONFIG["fig5_max_features"], len(self.feature_names))
        top_inds = np.argsort(mean_abs_shap)[::-1][:num_top]
        
        inter_vals = []
        for r in top_inds:
            for c in top_inds:
                if r != c:
                    inter_vals.append(np.abs(self.shap_interaction_values[:, r, c]).mean())
        max_inter = max(inter_vals) if inter_vals and max(inter_vals) > 0 else 1
        
        fig, axes = plt.subplots(num_top, num_top, figsize=(14, 14))
        plt.subplots_adjust(wspace=0.08, hspace=0.08, bottom=0.15, left=0.12, top=0.92)
        
        for row, feat_r_idx in enumerate(top_inds):
            for col, feat_c_idx in enumerate(top_inds):
                ax = axes[row, col]
                feat_row_name = self.feature_names[feat_r_idx]
                feat_col_name = self.feature_names[feat_c_idx]
                
                if row > col:
                    ax.set_xticks([])
                    ax.set_yticks([])
                    inter_val = np.abs(self.shap_interaction_values[:, feat_r_idx, feat_c_idx]).mean()
                    norm_val = 0.5 + 0.5 * (inter_val / max_inter)
                    bg_color = CONFIG["shap_cmap"](norm_val)
                    ax.set_facecolor(bg_color)
                    
                    text_color = 'white' if norm_val > 0.85 else 'black'
                    ax.text(0.5, 0.5, f"{inter_val:.3f}", ha='center', va='center',
                            fontweight='bold', fontsize=CONFIG["tick_fontsize"], color=text_color)
                
                else:
                    ax.set_facecolor('white')
                    ax.grid(True, color='lightgray', linestyle='-', linewidth=0.8)
                    
                    if row == col:
                        x_vals = self.shap_interaction_values[:, feat_c_idx, feat_c_idx]
                    else:
                        x_vals = self.shap_interaction_values[:, feat_r_idx, feat_c_idx] * 2
                    
                    nbins = 200
                    counts, bin_edges = np.histogram(x_vals, bins=nbins)
                    bin_indices = np.digitize(x_vals, bin_edges) - 1
                    bin_indices = np.clip(bin_indices, 0, nbins - 1)
                    
                    y_jitter = np.zeros_like(x_vals)
                    max_count = counts.max() if counts.max() > 0 else 1
                    
                    c_vals = self.X_train_processed[feat_row_name].values
                    
                    for b in range(nbins):
                        idx_in_bin = np.where(bin_indices == b)[0]
                        num_points = len(idx_in_bin)
                        if num_points > 1:
                            sorted_local_idx = np.argsort(c_vals[idx_in_bin])
                            actual_idx = idx_in_bin[sorted_local_idx]
                            width = (num_points / max_count) * 0.45
                            y_jitter[actual_idx] = np.linspace(-width, width, num_points)
                        else:
                            y_jitter[idx_in_bin] = 0
                        
                    c_vals = self.X_train_processed[feat_row_name].values
                    c_min, c_max = c_vals.min(), c_vals.max()
                    c_norm = (c_vals - c_min) / (c_max - c_min + 1e-8)
                    
                    ax.scatter(x_vals, y_jitter, c=c_norm, cmap=CONFIG["shap_cmap"],
                               s=8, alpha=0.7, edgecolors='none', vmin=0, vmax=1)
                    
                    ax.axhline(0, color='gray', linewidth=0.5, linestyle='-', alpha=0.3)
                    ax.axvline(0, color='gray', linewidth=0.8, linestyle='--')
                    ax.set_yticks([])
                    ax.set_ylim(-0.6, 0.6)
                    
                    if row == 0:
                        ax.xaxis.tick_top()
                        ax.tick_params(axis='x', labelsize=CONFIG["fig5_tick_fontsize"], pad=2)
                        ax.locator_params(axis='x', nbins=3)
                        plt.setp(ax.get_xticklabels(), rotation=0, ha='center')
                    else:
                        ax.set_xticks([])

                for spine in ax.spines.values():
                    spine.set_linewidth(0.8)
                    spine.set_color('#333333')

                if row == num_top - 1:
                    ax.set_xlabel(feat_col_name, fontsize=CONFIG["fig5_label_fontsize"], rotation=90,
                                  ha='center', va='top', labelpad=10)
                
                if col == 0:
                    ax.set_ylabel(feat_row_name, fontsize=CONFIG["fig5_label_fontsize"], rotation=0,
                                  ha='right', va='center', labelpad=15)

        fig.text(0.5, 0.96, "SHAP interaction value", ha='center', va='center', fontsize=CONFIG["title_fontsize"])

        cbar_ax = fig.add_axes([0.92, 0.25, 0.02, 0.5])
        sm_map = plt.cm.ScalarMappable(cmap=CONFIG["shap_cmap"], norm=plt.Normalize(vmin=0, vmax=1))
        sm_map.set_array([])
        cbar = fig.colorbar(sm_map, cax=cbar_ax)
        cbar.ax.tick_params(labelsize=CONFIG["fig5_cbar_fontsize"])
        cbar.set_label('Raw feature value / Mean |Interaction|', rotation=270, labelpad=20, fontsize=CONFIG["fig5_cbar_fontsize"])
        cbar.set_ticks([0, 1])
        cbar.set_ticklabels(['Low', 'High'])

        save_figure(fig, "Fig5_Combined_Matrix")

    def plot_figure_6(self):
        print("正在绘制图6：成对交互作用散点图...")
        num_features = len(self.feature_names)
        folder_name = "Fig6_Pairwise_Interactions"
        total_pairs = num_features * (num_features - 1) // 2
        
        from tqdm import tqdm
        pbar = tqdm(total=total_pairs, desc="图6 进度")
        plot_idx = 1
        for i in range(num_features):
            for j in range(num_features):
                if i >= j: continue
                
                feat_i_name = self.feature_names[i]
                feat_j_name = self.feature_names[j]
                
                x_vals = self.X_train_processed[feat_i_name].values
                y_vals = self.shap_interaction_values[:, i, j] * 2
                c_vals = self.X_train_processed[feat_j_name].values
                
                fig, ax = plt.subplots(figsize=(6, 5))
                scatter = ax.scatter(x_vals, y_vals, c=c_vals, cmap=CONFIG["shap_cmap"],
                                     s=30, alpha=0.8, edgecolors='none', zorder=2)
                
                ax.axhline(0, color='gray', linestyle='--', alpha=0.5, zorder=1)
                
                sorted_indices = np.argsort(x_vals)
                x_sorted = x_vals[sorted_indices]
                y_sorted = y_vals[sorted_indices]
                lowess_inter = sm.nonparametric.lowess(y_sorted, x_sorted, frac=0.3)
                
                sign_changes = np.diff(np.sign(lowess_inter[:, 1]))
                zero_crossings = np.where(sign_changes != 0)[0]
                
                x_min, x_max = ax.get_xlim()
                y_min, y_max = ax.get_ylim()
                x_span = x_max - x_min
                y_span = y_max - y_min
    
                for zc_idx, zc in enumerate(zero_crossings):
                    threshold_val = lowess_inter[zc, 0]
                    ax.axvline(x=threshold_val, color='orange', linestyle='--', alpha=0.9, zorder=3)
                    
                    if zc_idx % 2 == 0:
                        y_pos = y_min + 0.15 * y_span
                        ha_align = 'right'
                        x_offset = -0.02 * x_span
                    else:
                        y_pos = y_max - 0.15 * y_span
                        ha_align = 'left'
                        x_offset = 0.02 * x_span
    
                    if threshold_val + x_offset < x_min + 0.05 * x_span:
                        ha_align = 'left'
                        x_offset = 0.02 * x_span
                    elif threshold_val + x_offset > x_max - 0.05 * x_span:
                        ha_align = 'right'
                        x_offset = -0.02 * x_span
    
                    ax.text(threshold_val + x_offset, y_pos, f"{threshold_val:.2f}", rotation=90,
                            va='center', ha=ha_align, fontsize=CONFIG["tick_fontsize"],
                            bbox=dict(facecolor=(1, 1, 1, 0.7), edgecolor='none', pad=2), zorder=4)
    
                ax.set_xlabel(feat_i_name, fontsize=CONFIG["label_fontsize"])
                ax.set_ylabel("SHAP Interaction Value", fontsize=CONFIG["label_fontsize"])
                ax.set_title(f"{feat_i_name} × {feat_j_name}", fontsize=CONFIG["title_fontsize"])
                ax.grid(True, linestyle='--', alpha=0.3)
                
                divider = make_axes_locatable(ax)
                cax = divider.append_axes("right", size="5%", pad=0.0)
                cbar = fig.colorbar(scatter, cax=cax)
                cbar.set_label(feat_j_name, rotation=270, labelpad=15)
                
                clean_i_name = re.sub(r'[^\w]', '', feat_i_name)
                clean_j_name = re.sub(r'[^\w]', '', feat_j_name)
                filename = f"{plot_idx:02d}_{clean_i_name}_cross_{clean_j_name}"
                save_figure(fig, filename, subfolder=folder_name)
                plt.close(fig)
                plot_idx += 1
                pbar.update(1)
        pbar.close()

    def plot_figure_7(self):
        print(f"正在绘制图7：特征重要性与交互强度网络图...")
        mean_abs_shap = np.abs(self.shap_values_obj.values).mean(axis=0)
        sort_inds = np.argsort(mean_abs_shap)[::-1]
        num_nodes = min(CONFIG.get("fig7_max_features", 18), len(self.feature_names))
        top_inds = sort_inds[:num_nodes]
        top_names = [self.feature_names[i] for i in top_inds]

        G = nx.Graph()
        for idx, name in zip(top_inds, top_names):
            G.add_node(name, weight=mean_abs_shap[idx])
            
        for i in range(num_nodes):
            for j in range(i + 1, num_nodes):
                idx_i = top_inds[i]
                idx_j = top_inds[j]
                inter_val = np.abs(self.shap_interaction_values[:, idx_i, idx_j]).mean()
                G.add_edge(top_names[i], top_names[j], weight=inter_val)

        pos = nx.circular_layout(G)
        node_weights = [G.nodes[n]['weight'] for n in G.nodes]
        edge_weights = [G[u][v]['weight'] for u, v in G.edges]

        node_norm = (node_weights - np.min(node_weights)) / (np.max(node_weights) - np.min(node_weights) + 1e-8)
        edge_norm = (edge_weights - np.min(edge_weights)) / (np.max(edge_weights) - np.min(edge_weights) + 1e-8)

        node_sizes = 400 + 1500 * node_norm
        edge_widths = 1 + 7 * edge_norm

        node_cmap = mcolors.LinearSegmentedColormap.from_list("node_cmap", ["#E0F2F1", CONFIG["fig7_node_color"]])
        edge_cmap = mcolors.LinearSegmentedColormap.from_list("edge_cmap", ["#FCE4EC", CONFIG["fig7_edge_color"]])

        fig = plt.figure(figsize=(10, 10))
        ax_main = fig.add_axes([0.1, 0.15, 0.8, 0.8])
        
        edges = nx.draw_networkx_edges(G, pos, width=edge_widths, edge_color=edge_weights,
                                       edge_cmap=edge_cmap, alpha=0.8, ax=ax_main)
        nodes = nx.draw_networkx_nodes(G, pos, node_size=node_sizes, node_color=node_weights,
                                       cmap=node_cmap, edgecolors='gray', linewidths=1.5, ax=ax_main)

        for node, (x, y) in pos.items():
            offset_y = 0.12 + (node_sizes[top_names.index(node)] / 40000.0)
            if y >= 0:
                ax_main.text(x, y + offset_y, node, ha='center', va='bottom', fontsize=CONFIG["tick_fontsize"])
            else:
                ax_main.text(x, y - offset_y, node, ha='center', va='top', fontsize=CONFIG["tick_fontsize"])
        
        ax_main.set_title("Feature Impact & Interaction Network", fontsize=CONFIG["title_fontsize"])
        ax_main.axis('off')

        cax1 = fig.add_axes([0.15, 0.10, 0.3, 0.015])
        sm_node = plt.cm.ScalarMappable(cmap=node_cmap, norm=plt.Normalize(vmin=np.min(node_weights), vmax=np.max(node_weights)))
        sm_node.set_array([])
        cbar1 = fig.colorbar(sm_node, cax=cax1, orientation='horizontal')
        cbar1.set_label('Importance (Vimp)', fontsize=CONFIG["label_fontsize"])
        cbar1.ax.tick_params(labelsize=CONFIG["tick_fontsize"] - 2)
        
        cax2 = fig.add_axes([0.55, 0.10, 0.3, 0.015])
        sm_edge = plt.cm.ScalarMappable(cmap=edge_cmap, norm=plt.Normalize(vmin=np.min(edge_weights), vmax=np.max(edge_weights)))
        sm_edge.set_array([])
        cbar2 = fig.colorbar(sm_edge, cax=cax2, orientation='horizontal')
        cbar2.set_label('Interaction Intensity (Vint)', fontsize=CONFIG["label_fontsize"])
        cbar2.ax.tick_params(labelsize=CONFIG["tick_fontsize"] - 2)

        save_figure(fig, "Fig7_Impact_Interaction_Network")

    def plot_figure_8(self):
        print("正在绘制图8：SHAP 特征综合热力图...")
        fig = plt.figure(figsize=CONFIG["fig8_figsize"])
        shap.plots.heatmap(
            self.shap_values_obj,
            max_display=CONFIG["fig8_max_display"],
            cmap=CONFIG["shap_cmap"],
            show=False
        )
        
        current_fig = plt.gcf()
        for ax in current_fig.axes:
            ax.tick_params(labelsize=CONFIG["tick_fontsize"])
            if ax.get_xlabel():
                ax.set_xlabel(ax.get_xlabel(), fontsize=CONFIG["label_fontsize"], fontfamily=CONFIG["font_family_en"])
            if ax.get_ylabel():
                ylabel_text = ax.get_ylabel()
                if "f(x)" in ylabel_text.lower():
                    ax.set_ylabel("f(x)", fontsize=CONFIG["label_fontsize"], fontfamily=CONFIG["font_family_en"], fontstyle='italic')
                else:
                    ax.set_ylabel(ylabel_text, fontsize=CONFIG["label_fontsize"], fontfamily=CONFIG["font_family_en"])
                    
        save_figure(current_fig, "Fig8_SHAP_Heatmap")

    def plot_figure_9(self):
        print("正在绘制图9：各样本 SHAP 单变量推演力图...")
        folder_name = "Fig9_Force_Plots"
        max_instances = min(len(self.X), CONFIG["fig9_max_instances"])
        
        from tqdm import tqdm
        for i in tqdm(range(max_instances), desc="图9 进度"):
            preprocessor = Pipeline(self.pipeline.steps[:-1])
            X_all_transformed = preprocessor.transform(self.X)
            X_all_processed = pd.DataFrame(X_all_transformed, columns=self.feature_names)
            
            shap_values_all = self.explainer.shap_values(X_all_processed, check_additivity=False)
            base_value = self.explainer.expected_value
            
            if isinstance(base_value, (list, np.ndarray)):
                base_value = base_value[0]
                
            instance_features = X_all_processed.iloc[i, :].round(3)
            
            shap_fig = shap.force_plot(
                base_value,
                shap_values_all[i, :],
                instance_features,
                matplotlib=True,
                show=False
            )
            
            fig = shap_fig if shap_fig is not None else plt.gcf()
            fig.set_size_inches(CONFIG["fig9_figsize"])
            ax = fig.gca()
            
            feature_texts = [t for t in ax.texts if '=' in t.get_text()]
            feature_texts.sort(key=lambda t: t.get_position()[0])
            
            lines = ax.get_lines()
            x_span = ax.get_xlim()[1] - ax.get_xlim()[0]
            x_threshold = x_span * 0.15
            y_span = ax.get_ylim()[1] - ax.get_ylim()[0]
            step = y_span * 0.12
            
            levels = [0.0, -step, -step*2, -step*3, -step*4, -step*5]
            last_x_at_level = {lvl: -float('inf') for lvl in levels}
            min_y_attained = ax.get_ylim()[0]
            
            for text_obj in feature_texts:
                x, y = text_obj.get_position()
                chosen_level = levels[0]
                for lvl in levels:
                    if x - last_x_at_level[lvl] > x_threshold:
                        chosen_level = lvl
                        break
                
                last_x_at_level[chosen_level] = x
                new_y = y + chosen_level
                min_y_attained = min(min_y_attained, new_y)
                
                if chosen_level != 0.0:
                    text_obj.set_position((x, new_y))
                    for line in lines:
                        xdata = line.get_xdata()
                        ydata = line.get_ydata()
                        if len(xdata) == 2 and abs(xdata[0] - x) < 1e-3 and abs(xdata[1] - x) < 1e-3:
                            if abs(ydata[0] - y) < abs(ydata[1] - y):
                                ydata[0] = new_y
                            else:
                                ydata[1] = new_y
                            line.set_ydata(ydata)
            
            ax.set_ylim(bottom=min_y_attained - step)
    
            top_texts = [t for t in ax.texts if 'base value' in t.get_text() or 'f(x)' in t.get_text()]
            top_texts.sort(key=lambda t: t.get_position()[0])
            for idx in range(1, len(top_texts)):
                prev_x, prev_y = top_texts[idx-1].get_position()
                curr_x, curr_y = top_texts[idx].get_position()
                if abs(curr_x - prev_x) < x_threshold and abs(curr_y - prev_y) < step:
                    top_texts[idx].set_position((curr_x, curr_y + step))
    
            pos_color = CONFIG["fig9_color_pos"]
            neg_color = CONFIG["fig9_color_neg"]
            target_pos = mcolors.to_rgb("#FF0051")
            target_neg = mcolors.to_rgb("#008BFB")
            
            def match_color(c):
                if c is None: return None
                try:
                    c_rgb = mcolors.to_rgb(c)
                    if sum((a - b)**2 for a, b in zip(c_rgb, target_pos)) < 0.05:
                        return pos_color
                    if sum((a - b)**2 for a, b in zip(c_rgb, target_neg)) < 0.05:
                        return neg_color
                except:
                    pass
                return None
    
            for obj in ax.findobj():
                if hasattr(obj, 'get_color') and hasattr(obj, 'set_color'):
                    new_c = match_color(obj.get_color())
                    if new_c: obj.set_color(new_c)
                
                if hasattr(obj, 'get_facecolor') and hasattr(obj, 'get_facecolor'):
                    fc = obj.get_facecolor()
                    if isinstance(fc, np.ndarray) and fc.size >= 3:
                        fc = fc[0] if fc.ndim == 2 else fc
                    new_c = match_color(fc)
                    if new_c: obj.set_facecolor(new_c)
                
                if hasattr(obj, 'get_edgecolor') and hasattr(obj, 'set_edgecolor'):
                    ec = obj.get_edgecolor()
                    if isinstance(fc, np.ndarray) and ec.size >= 3:
                        ec = ec[0] if ec.ndim == 2 else ec
                    new_c = match_color(ec)
                    if new_c: obj.set_edgecolor(new_c)
            
            filename = f"Instance_{i+1:04d}_Force_Plot"
            save_figure(fig, filename, subfolder=folder_name)
            plt.close(fig)

    def plot_figure_10(self):
        print("正在绘制图10：二维PDP偏依赖图 (运算较慢，请等待)...")
        mean_abs_shap = np.abs(self.shap_values_obj.values).mean(axis=0)
        sort_inds = np.argsort(mean_abs_shap)[::-1]
        X_bg_median = self.X_train_processed.median().values
        grid_resolution = 50
        cmap = CONFIG["shap_cmap"]
        
        from tqdm import tqdm
        for rank, idx1 in tqdm(enumerate(sort_inds), total=len(sort_inds), desc="图10 进度"):
            feat1 = self.feature_names[idx1]
            clean_feat1 = re.sub(r'[^\w]', '', feat1)
            folder_name = f"Fig10_2D_PDP/{rank+1:02d}_{clean_feat1}"
            x1_min, x1_max = self.X_train_processed[feat1].min(), self.X_train_processed[feat1].max()
            x1_grid = np.linspace(x1_min, x1_max, grid_resolution)
            
            for idx2 in range(len(self.feature_names)):
                if idx1 == idx2: continue
                feat2 = self.feature_names[idx2]
                x2_min, x2_max = self.X_train_processed[feat2].min(), self.X_train_processed[feat2].max()
                x2_grid = np.linspace(x2_min, x2_max, grid_resolution)
                
                X1, X2 = np.meshgrid(x1_grid, x2_grid)
                grid_points = np.tile(X_bg_median, (grid_resolution * grid_resolution, 1))
                grid_points[:, idx1] = X1.ravel()
                grid_points[:, idx2] = X2.ravel()
                
                preds = self.model.predict(grid_points).reshape(grid_resolution, grid_resolution)
                fig, ax = plt.subplots(figsize=(7, 6))
                cf = ax.contourf(X1, X2, preds, levels=60, cmap=cmap, alpha=0.9)
                
                q1, med, q3 = np.percentile(preds, [25, 50, 75])
                cs1 = ax.contour(X1, X2, preds, levels=[q1], colors=[CONFIG["fig10_line_color_q1"]], linestyles=['--'], linewidths=1.5)
                cs2 = ax.contour(X1, X2, preds, levels=[med], colors=[CONFIG["fig10_line_color_med"]], linestyles=['-'], linewidths=1.5)
                cs3 = ax.contour(X1, X2, preds, levels=[q3], colors=[CONFIG["fig10_line_color_q3"]], linestyles=['--'], linewidths=1.5)
                
                p_max_val, p_min_val = preds.max(), preds.min()
                max_idx = np.unravel_index(np.argmax(preds), preds.shape)
                min_idx = np.unravel_index(np.argmin(preds), preds.shape)
                
                ax.scatter(X1[max_idx], X2[max_idx], marker='*', color='orange', s=120, edgecolors='black', linewidth=0.5, zorder=5)
                ax.scatter(X1[min_idx], X2[min_idx], marker='o', color='#00BCD4', s=60, edgecolors='white', linewidth=0.5, zorder=5)
                
                h_mid, w_mid = grid_resolution // 2, grid_resolution // 2
                quadrants = {
                    'lower left': preds[:h_mid, :w_mid], 'lower right': preds[:h_mid, w_mid:],
                    'upper left': preds[h_mid:, :w_mid], 'upper right': preds[h_mid:, w_mid:]
                }
                best_loc = min(quadrants, key=lambda k: np.std(quadrants[k]))
    
                cbar = fig.colorbar(cf, ax=ax, pad=0.03)
                cbar.ax.tick_params(labelsize=CONFIG["tick_fontsize"])
                
                l1 = mlines.Line2D([], [], color=CONFIG["fig10_line_color_q1"], linestyle='--', label=f'Q1: {q1:.2f}')
                l2 = mlines.Line2D([], [], color=CONFIG["fig10_line_color_med"], linestyle='-', label=f'Median: {med:.2f}')
                l3 = mlines.Line2D([], [], color=CONFIG["fig10_line_color_q3"], linestyle='--', label=f'Q3: {q3:.2f}')
                p_max = ax.scatter([], [], marker='*', color='orange', s=100, edgecolors='black', label=f'Max: {p_max_val:.2f}')
                p_min = ax.scatter([], [], marker='o', color='#00BCD4', s=60, edgecolors='white', label=f'Min: {p_min_val:.2f}')
                
                leg = ax.legend(handles=[l1, l2, l3, p_max, p_min], loc=best_loc,
                                fontsize=CONFIG["label_fontsize"] - 2,
                                frameon=True, facecolor=(1, 1, 1, 0.8), edgecolor='none')
                
                for text in leg.get_texts(): text.set_color('black')
                
                ax.set_xlabel(feat1, fontsize=CONFIG["label_fontsize"])
                ax.set_ylabel(feat2, fontsize=CONFIG["label_fontsize"])
                ax.tick_params(axis='both', labelsize=CONFIG["tick_fontsize"])
                
                clean_feat2 = re.sub(r'[^\w]', '', feat2)
                save_figure(fig, f"{clean_feat1}_vs_{clean_feat2}", subfolder=folder_name)
                plt.close(fig)

print("✅ 第四段：核心分析类定义完成")

✅ 第四段：核心分析类定义完成


In [5]:
# ====================== 第五段：初始化 + 数据 + 训练模型 ======================
analyzer = XGBoostXAIAnalyzer()

# 读取数据
analyzer.prepare_data()

# 训练XGBoost模型
analyzer.train_xgboost(analyzer.X_train, analyzer.y_train)

# 计算SHAP值（必须运行，后面所有图都依赖它）
analyzer.calculate_shap()

print("✅ 第五段：数据加载、模型训练、SHAP计算完成")

正在读取数据表...
数据读取完毕。特征数量: 26
XGBoost 最优参数: {'reg__subsample': 0.7, 'reg__reg_lambda': 1.5, 'reg__reg_alpha': 0, 'reg__n_estimators': 800, 'reg__max_depth': 6, 'reg__learning_rate': 0.1, 'reg__colsample_bytree': 0.85}
正在计算 SHAP 值 (交互作用计算较耗时)...
SHAP 计算完成。
✅ 第五段：数据加载、模型训练、SHAP计算完成


In [6]:
# ====================== 第六段：绘图（按需运行，不想画的注释掉） ======================
analyzer.plot_figure_1()   # 模型评估图
analyzer.plot_figure_2()   # 全局特征重要性
analyzer.plot_figure_3()   # 单特征依赖图
analyzer.plot_figure_4()   # 主效应 vs 交互效应
analyzer.plot_figure_5()   # 交互矩阵图
analyzer.plot_figure_6()   # 成对交互散点图
analyzer.plot_figure_7()   # 特征交互网络
analyzer.plot_figure_8()   # SHAP热力图
analyzer.plot_figure_9()   # 样本力图
analyzer.plot_figure_10() # 二维PDP（最慢，建议最后跑）

print(f"✅ 绘图完成！所有图片保存在文件夹：{CONFIG['output_dir']}")

🔥 整体绘图进度:   0%|          | 0/10 [00:00<?, ?it/s]

正在绘制图1：模型评估组合图...


🔥 整体绘图进度:  10%|█         | 1/10 [00:02<00:24,  2.75s/it, 当前=第1张图]

正在绘制图2：全局特征贡献度分析图...


🔥 整体绘图进度:  20%|██        | 2/10 [00:06<00:28,  3.52s/it, 当前=第2张图]

...

正在绘制图9：各样本 SHAP 单变量推演力图...



🔥 整体绘图进度:  90%|█████████ | 9/10 [08:24<01:36, 96.18s/it, 当前=第9张图]

正在绘制图10：二维PDP偏依赖图 (运算较慢，请等待)...



🔥 整体绘图进度: 100%|██████████| 10/10 [14:49<00:00, 88.97s/it, 当前=第10张图] 

✅ 绘图完成！所有图片保存在文件夹：PM25_纯街景_运行结果
